In [1]:
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

# Load prepared train/test data
train_df = pd.read_csv("C:/Users/BLACKBOX/Desktop/Research Data/Iyenshi/Coding Assignment/train_set.csv")
test_df = pd.read_csv("C:/Users/BLACKBOX/Desktop/Research Data/Iyenshi/Coding Assignment/test_set.csv")

if "Attack Type_encoded" not in train_df.columns or "Attack Type_encoded" not in test_df.columns:
    raise ValueError("The train/test files must already contain 'Attack Type_encoded'.")

feature_cols = [c for c in train_df.columns if c not in ["Attack Type", "Attack Type_encoded"]]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df["Attack Type_encoded"]
y_test = test_df["Attack Type_encoded"]

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Target classes:", sorted(pd.Series(y_train).unique().tolist()))

save_dir = Path("saved/models")
save_dir.mkdir(parents=True, exist_ok=True)
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)
results_file = results_dir / "results.csv"

# Helper to append one row per model run to results.csv

def append_result_row(metrics):
    row = pd.DataFrame([metrics])
    row.to_csv(results_file, mode="a", header=not results_file.exists(), index=False)

# Compute class weights from the training labels
classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print("\nClass weights:")
print(class_weight_dict)


def evaluate_model(model, name, X_train, X_test, y_train, y_test):
    try:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        metrics = {
            "model": name,
            "status": "success",
            "accuracy": accuracy_score(y_test, pred),
            "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0),
            "error": "",
        }
        append_result_row(metrics)

        print(f"\n{name}")
        print("=" * 40)
        print(classification_report(y_test, pred, zero_division=0))
        print("Confusion Matrix:\n", confusion_matrix(y_test, pred))

        model_path = save_dir / f"{name.lower().replace(' ', '_')}.pkl"
        joblib.dump(model, model_path)
        print(f"Saved model: {model_path}")

        return model, metrics
    except Exception as e:
        metrics = {
            "model": name,
            "status": "failed",
            "accuracy": np.nan,
            "precision_macro": np.nan,
            "recall_macro": np.nan,
            "f1_macro": np.nan,
            "error": str(e),
        }
        append_result_row(metrics)
        print(f"\n{name} failed: {e}")
        return None, metrics

Training shape: (3713654, 29)
Test shape: (928414, 29)
Target classes: [0, 1, 2, 3]

Class weights:
{np.int64(0): np.float64(8.28940625), np.int64(1): np.float64(1.480617021106938), np.int64(2): np.float64(0.3505923807308408), np.int64(3): np.float64(2.8436984081768926)}


In [2]:
# 1. Train Random Forest with hyperparameter tuning and class weights
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

try:
    rf_search = RandomizedSearchCV(
        RandomForestClassifier(random_state=42, class_weight=class_weight_dict),
        param_distributions=rf_param_grid,
        n_iter=10,
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        random_state=42,
    )
    rf_search.fit(X_train, y_train)
    best_rf = rf_search.best_estimator_
    rf_model, rf_metrics = evaluate_model(best_rf, "Random Forest", X_train, X_test, y_train, y_test)
    joblib.dump(rf_search, save_dir / "random_forest_search.pkl")
except Exception as e:
    print(f"Random Forest setup failed: {e}")



Random Forest
              precision    recall  f1-score   support

           0       0.26      0.99      0.42     28000
           1       0.97      0.96      0.96    156761
           2       0.99      0.88      0.93    662033
           3       0.88      0.85      0.87     81620

    accuracy                           0.90    928414
   macro avg       0.78      0.92      0.79    928414
weighted avg       0.95      0.90      0.92    928414

Confusion Matrix:
 [[ 27648     35    269     48]
 [  2278 150029   4318    136]
 [ 64148   5182 583747   8956]
 [ 10451     87   1449  69633]]
Saved model: saved\models\random_forest.pkl


In [3]:
try:
    import xgboost as xgb
    from xgboost import XGBClassifier

    xgb_params = {
        "objective": "multi:softprob",
        "num_class": len(np.unique(y_train)),
        "eval_metric": "mlogloss",
        "random_state": 42,
        "tree_method": "hist",
        "device": "cpu",
    }

    xgb_param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }

    xgb_search = RandomizedSearchCV(
        estimator=XGBClassifier(**xgb_params),
        param_distributions=xgb_param_grid,
        n_iter=8,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=1,
        verbose=2,
    )

    xgb_search.fit(X_train, y_train)
    best_xgb = xgb_search.best_estimator_
    xgb_model, xgb_metrics = evaluate_model(best_xgb, "XGBoost", X_train, X_test, y_train, y_test)
    joblib.dump(best_xgb, save_dir / "xgboost_best_model.pkl")
    joblib.dump(xgb_search, save_dir / "xgboost_search.pkl")
except Exception as e:
    print(f"XGBoost failed: {e}")


Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=100, subsample=0.8; total time=  21.7s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=100, subsample=0.8; total time=  21.7s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=100, subsample=0.8; total time=  21.5s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.8; total time=  33.9s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.8; total time=  33.7s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.8; total time=  33.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=5, n_estimators=200, subsample=0.8; total time=  41.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=5, n_estimators=200, subsample=0.8; total time=  41.7s
[CV] END colsam

In [4]:
try:
    from lightgbm import LGBMClassifier

    lgbm_params = {
        "objective": "multiclass",
        "num_class": len(np.unique(y_train)),
        "random_state": 42,
        "device": "cpu",
        "verbosity": -1,
        "class_weight": class_weight_dict,
    }

    lgbm_param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }

    lgbm_search = RandomizedSearchCV(
        estimator=LGBMClassifier(**lgbm_params),
        param_distributions=lgbm_param_grid,
        n_iter=8,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=1,
        verbose=2,
    )

    lgbm_search.fit(X_train, y_train)
    best_lgbm = lgbm_search.best_estimator_
    lgbm_model, lgbm_metrics = evaluate_model(best_lgbm, "LightGBM", X_train, X_test, y_train, y_test)
    joblib.dump(best_lgbm, save_dir / "lightgbm_best_model.pkl")
    joblib.dump(lgbm_search, save_dir / "lightgbm_search.pkl")
except Exception as e:
    print(f"LightGBM failed: {e}")


Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=100, subsample=0.8; total time=  19.9s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=100, subsample=0.8; total time=  14.6s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=100, subsample=0.8; total time=  14.9s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.8; total time=  19.9s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.8; total time=  18.4s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.8; total time=  18.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=5, n_estimators=200, subsample=0.8; total time=  27.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=5, n_estimators=200, subsample=0.8; total time=  28.3s
[CV] END colsam

In [5]:
# Final comparison and result saving
results_df = pd.DataFrame()
if results_file.exists():
    results_df = pd.read_csv(results_file)

if not results_df.empty:
    results_df = results_df.sort_values(by="f1_macro", ascending=False, na_position="last").reset_index(drop=True)
    print("\nModel comparison (sorted by F1 Macro)")
    print(results_df)
    results_df.to_csv(results_file, index=False)

# Save the best successful model based on F1 macro
successful = results_df[results_df["status"] == "success"]
if not successful.empty:
    best_result = successful.sort_values(by="f1_macro", ascending=False).iloc[0]
    best_model_name = best_result["model"]
    if best_model_name == "Random Forest" and "rf_model" in locals():
        best_model = rf_model
    elif best_model_name == "XGBoost" and "xgb_model" in locals():
        best_model = xgb_model
    elif best_model_name == "LightGBM" and "lgbm_model" in locals():
        best_model = lgbm_model
    else:
        best_model = None

    if best_model is not None:
        joblib.dump(best_model, save_dir / "best_model.pkl")
        print(f"\nBest model saved: {save_dir / 'best_model.pkl'}")

# Optional feature importance plots for successful models
for name, model in [("Random Forest", rf_model if "rf_model" in locals() else None), ("XGBoost", xgb_model if "xgb_model" in locals() else None), ("LightGBM", lgbm_model if "lgbm_model" in locals() else None)]:
    if model is not None and hasattr(model, "feature_importances_"):
        importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(20)
        plt.figure(figsize=(10, 5))
        importances.plot(kind="bar")
        plt.title(f"{name} Feature Importance")
        plt.ylabel("Importance")
        plt.tight_layout()
        plt.savefig(results_dir / f"{name.lower().replace(' ', '_')}_feature_importance.png")
        plt.close()

print("\nAll results saved to:", results_dir)



Model comparison (sorted by F1 Macro)
           model   status  accuracy  precision_macro  recall_macro  f1_macro  \
0        XGBoost  success  0.951093         0.964300      0.838673  0.890479   
1  Random Forest  success  0.895136         0.776025      0.919842  0.794901   
2       LightGBM  success  0.866145         0.722605      0.906769  0.756747   

   error  
0    NaN  
1    NaN  
2    NaN  

Best model saved: saved\models\best_model.pkl

All results saved to: results
